# Year-to-date US stock aggregates from Massive flat files

Download the `day_aggs` flat files for the current year-to-date range with
`MassiveFlatFiles`, then use them: per-ticker YTD returns and the market's top
movers.

Flat-files credentials are read from the environment (issued in the Massive
Dashboard, separate from `MASSIVE_API_KEY`):

```bash
export MASSIVE_S3_ACCESS_KEY=...
export MASSIVE_S3_SECRET_KEY=...
```

Note: downloaded files are reused as an on-disk cache (historical flat files
are immutable), so re-running is fast; pass `overwrite=True` to force a fresh
fetch of a recent day that may still be finalizing.

In [ ]:
import os
from datetime import date

import polars as pl
from dotenv import load_dotenv

from xpectral.data import MassiveFlatFiles

assert os.getenv("MASSIVE_S3_ACCESS_KEY") and os.getenv("MASSIVE_S3_SECRET_KEY"), (
    "set MASSIVE_S3_ACCESS_KEY and MASSIVE_S3_SECRET_KEY first"
)

pl.Config.set_tbl_rows(12)

## Download year-to-date

`day_aggs` is one row per ticker per day for the whole US market -- the smallest,
most practical dataset for a full-year pull. Missing days (weekends/holidays) are
skipped automatically.

In [ ]:
flat_files = MassiveFlatFiles()

start = date(date.today().year, 1, 1)
end = date.today()
print(f"fetching day_aggs {start}..{end} ...")

df = flat_files.get_flat_files(
    dataset="day_aggs",
    from_=start,
    to=end,
    prefix="us_stocks_sip",
    overwrite=False
).collect()

df.shape

In [ ]:
print("rows:   ", df.height)
print("tickers:", df.get_column("ticker").n_unique())
print(
    "dates:  ",
    df.get_column("window_start").dt.date().min(),
    "->",
    df.get_column("window_start").dt.date().max(),
)
df.head()

## A few tickers

Pivot closing prices for a handful of names into a wide, date-indexed table.

In [ ]:
TICKERS = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]

closes = (
    df.filter(pl.col("ticker").is_in(TICKERS))
    .select("window_start", "ticker", "close")
    .pivot(index="window_start", on="ticker", values="close")
    .sort("window_start")
)
closes.tail()

## YTD returns

First close of the year vs the latest close, per ticker.

In [ ]:
def ytd_return(frame: pl.DataFrame, tickers: list[str] | None = None) -> pl.DataFrame:
    if tickers is not None:
        frame = frame.filter(pl.col("ticker").is_in(tickers))
    return frame.group_by("ticker").agg(
        first_close=pl.col("close").sort_by("window_start").first(),
        last_close=pl.col("close").sort_by("window_start").last(),
    ).with_columns(ytd_return=pl.col("last_close") / pl.col("first_close") - 1)


ytd_return(df, TICKERS).sort("ytd_return", descending=True)

## Market-wide top movers

Because we pulled the whole market, we can rank YTD performance across every
ticker. Drop very low-priced names to avoid penny-stock noise.

In [ ]:
movers = (
    ytd_return(df)
    .filter(pl.col("first_close") >= 5)  # drop sub-$5 noise
    .sort("ytd_return", descending=True)
)

print("Top 10 YTD gainers")
print(movers.head(10))
print("\nTop 10 YTD losers")
print(movers.tail(10).reverse())